In [29]:
import sys
from pathlib import Path

# go to project root
PROJECT_ROOT = Path().resolve().parents[1]

sys.path.append(str(PROJECT_ROOT))

In [26]:
import pandas as pd

def download_cash_rate():
    url = "https://www.rba.gov.au/statistics/tables/csv/f1.1-data.csv"

    df = pd.read_csv(url, skiprows=1)
    df.columns = [c.strip() for c in df.columns]

    # keep only Date + Cash Rate
    df = df.rename(columns={
        "Title": "Date",
        "Cash Rate Target": "CashRate"
    })[["Date", "CashRate"]]

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["CashRate"] = pd.to_numeric(df["CashRate"], errors="coerce")

    df = df.dropna().sort_values("Date")
    df = df[df["Date"] >= "2000-01-01"]

    return df
# Since the data is montly, now we need to converge it to quarterly data.
df = download_cash_rate()
df.head()

/var/folders/39/270n5qyx79g5v2pbpttsls1r0000gn/T/ipykernel_5729/3840246727.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


,Date,CashRate
374,2000-01-31,5.00
375,2000-02-29,5.48
376,2000-03-31,5.50
377,2000-04-30,5.72
378,2000-05-31,5.98


In [27]:
# Convert data from monthly to quarterly by taking the average of each quarter.
def to_quarterly(df):
    df_q = (
        df.set_index("Date")
          .resample("QE")
          .mean()
          .reset_index()
    )
    return df_q


In [32]:
from src.utils.paths import RAW_DIR

def save_cash_rate_quarterly(df_q):
    # make sure folder exists
    RAW_DIR.mkdir(parents=True, exist_ok=True)

    # define path
    output_path = RAW_DIR / "cash_rate_quarterly.csv"

    # save
    df_q.to_csv(output_path, index=False)

    print(f"Saved to {output_path}")

df = download_cash_rate()
df_q = to_quarterly(df)

save_cash_rate_quarterly(df_q)

Saved to /Users/minhquan/Documents/GitHub/MAST90106 project/data/raw/cash_rate_quarterly.csv


/var/folders/39/270n5qyx79g5v2pbpttsls1r0000gn/T/ipykernel_5729/3840246727.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
